In [ ]:
import math
import numpy as np 
import matplotlib.pyplot as plt 
%matplotlib inline

In [ ]:
# h = 0.000000001
# x = 2/3
# (f(x+h) - f(x)) / h
print("hi")

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op = '', label='', grad=0.0):
        self.data = data
        self._prev = set(_children)
        self.grad = 0.0
        self._backward = lambda: None
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data}, children ={self._prev}, op='{self._op}', label='{self.label}')"
    
    def __add__(self, other):
        out = Value(self.data + other.data, _children=(self, other), _op='+', label='')
        def _backward():
            self.grad = out.grad
            other.grad = out.grad
        out._backward = _backward
        return out
    
    def __mul__(self, other):
        out = Value(self.data * other.data, _children=(self, other), _op='*', label='')
        def _backward():
            self.grad = out.grad * other.data
            other.grad = out.grad * self.data
        out._backward = _backward
        # print(type(out._backward))
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, _children=(self,), _op='tanh', label='')
        def _backward():
            self.grad  = out.grad * (1 - t**2)
        out._backward = _backward
        return out

a = Value(2.0, label = 'a')
b = Value(-3.0, label = 'b')
c = Value(10.0, label = 'c')
e = a*b; e.label = 'e'
d = e + c; d.label = 'd'
f = Value(-2.0, label = 'f')
L = d*f; L.label = 'L'

In [ ]:
print(a._backward)

In [ ]:
from graphviz import Digraph

def trace(root):
    # build a set of all nodes and edges in the graph
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        # for any value in the graph, create a reectangular ('record') node for it
        dot.node(name = uid, label = "{%s | data %.4f | grad %.4f}" % (n.label, n.data, n.grad), shape='record')
        if n._op:
            # if this node is result of some operation, create an op node
            dot.node(name = uid + n._op, label = n._op)
            # Now connect this node to the op node
            dot.edge(uid + n._op, uid)
    for n1, n2 in edges:
        # connect n1 to op node of n2
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot 

# def back_prop(root):
#     nodes, edges = trace(root)
#     # call _backward on all nodes in reverse topological order
    


In [ ]:
# def lol():
#     h=0.001
#     a = Value(2.0, label = 'a')
#     b = Value(-3.0, label = 'b')
#     c = Value(10.0, label = 'c')
#     e = a*b; e.label = 'e'
#     d = e + c; d.label = 'd'
#     f = Value(-2.0, label = 'f')
#     L = d*f; L.label = 'L'
#     L1 = L.data


#     a = Value(2.0, label = 'a')
#     # a.data+=h
#     b = Value(-3.0, label = 'b')
#     # b.data+=h
#     c = Value(10.0, label = 'c')
#     # c.data+=h
#     e = a*b; e.label = 'e'
#     # e.data+=h
#     d = e + c; d.label = 'd'
#     # d.data+=h
#     f = Value(-2.0, label = 'f')
#     # f.data+=h
#     L = d*f; L.label = 'L'
#     # L.data+=h

#     L2 = L.data
#     print((L2-L1)/h)

# lol()

In [ ]:
# a.grad = 6.0
# b.grad = -4.0
# c.grad = -2.0
# d.grad = -2.0
# e.grad = -2.0
# f.grad = 4.0
L.grad = 1.0 
L._backward()
d._backward()
e._backward()

In [ ]:
draw_dot(L) 

In [ ]:
a.data += 0.0001 * a.grad
b.data += 0.0001 * b.grad
c.data += 0.0001 * c.grad
# d.data += 0.0001 * d.grad
# e.data += 0.0001 * e.grad
f.data += 0.0001 * f.grad
# L.data -= 0.001 * L.grad

e = a*b; 
d = e + c
L = d*f

L.data

In [ ]:
x1 = Value(2.0, label = 'x1')
x2 = Value(0.0, label = 'x2')
# weights
w1 = Value(-3.0, label = 'w1')
w2 = Value(1.0, label = 'w2')
b = Value(6.8813735870195432, label = 'b')
# x1*w1 + x2*w2 + b
x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'

In [ ]:
o.grad = 1.0
# grad_cal(o)

In [ ]:
draw_dot(o)

In [ ]:
o.grad = 1.0

In [ ]:
n.grad = 1 - (o.data)**2

In [ ]:
b.grad = 0.5
x1w1x2w2.grad = 0.5

In [ ]:
x1w1.grad= 0.5
x2w2.grad= 0.5

In [ ]:
x1.grad = 0.5 * w1.data
w1.grad = 0.5 * x1.data

In [ ]:
x2.grad = 0.5 * w2.data
w2.grad = 0.5 * x2.data

In [ ]:
draw_dot(o)